# Final project: PageIndex (vectorless tree retrieval) on the two books

Same corpus, same ten questions and same answering model (`gemini-2.5-flash`) as the Assignment 5 classic-RAG run. Only the retrieval method changes.

**How PageIndex retrieves:** (1) the book becomes a tree of its own headings; each node gets a short summary. (2) For each question the model reads the tree (titles and summaries, no vectors) and picks the nodes that should hold the answer. (3) The text of those nodes goes to the model, with the same answer prompt as A5.

Run the cells top to bottom. Needs the Colab secret `GEMINI_API_KEY` (Notebook access ON). Upload the two corpus files when asked.

In [ ]:
# Step 1: install PageIndex. Its markdown parser builds the heading tree.
!pip install -q -U pageindex google-genai
import pageindex, importlib.metadata as md_
print("pageindex", md_.version("pageindex"))

In [ ]:
# Step 2: settings. Everything the comparison depends on lives here.
CHAT_MODEL = "gemini-2.5-flash"   # same as A5: used for summaries, tree search AND the answer
MAX_NODES  = 4                    # nodes the search may pick (A5 retrieved 4 chunks)
SUMMARY_WORDS = 40                # length of each node summary
MAX_NODE_CHARS = 12000            # safety cap on one node's text sent to the answer prompt

import time, json, re, os
from google import genai
from google.genai import types
from google.colab import userdata
api_key = userdata.get("GEMINI_API_KEY")
assert api_key, "No GEMINI_API_KEY secret (check Notebook access is ON)."
client = genai.Client(api_key=api_key)

def with_retry(fn, *a, **kw):
    for wait in [2, 5, 10, 20, 40, None]:
        try:
            return fn(*a, **kw)
        except Exception as e:
            if wait is None: raise
            print(f"  retry in {wait}s ({str(e)[:80]})"); time.sleep(wait)

USAGE = {"in": 0, "out": 0, "thought": 0, "calls": 0}
def llm(prompt, json_mode=False):
    """One Gemini call. Returns (text, prompt_tokens, answer_tokens, thought_tokens)."""
    cfg = types.GenerateContentConfig(response_mime_type="application/json") if json_mode else None
    r = with_retry(client.models.generate_content, model=CHAT_MODEL, contents=prompt, config=cfg)
    u = r.usage_metadata
    tin, tout, tth = u.prompt_token_count or 0, u.candidates_token_count or 0, getattr(u, "thoughts_token_count", 0) or 0
    return r.text, tin, tout, tth
print("Connected.", CHAT_MODEL)

In [ ]:
# Step 3: upload the two corpus files (Book1_..., Book2_...).
from google.colab import files
up = files.upload()
BOOKS = sorted(up)                      # Book1 sorts before Book2
for f in BOOKS:
    open(f, "wb").write(up[f])
    print(f, f"{len(up[f]):,} bytes")

In [ ]:
# Step 4: build the tree for each book with PageIndex's markdown parser (no LLM, no vectors).
from pageindex.page_index_md import md_to_tree

def flatten(nodes, path=()):
    out = []
    for n in nodes:
        p = path + (n["title"],)
        out.append({"node_id": n["node_id"], "title": n["title"], "path": " > ".join(p),
                    "text": n.get("text", ""), "line": n.get("line_num"), "depth": len(p) - 1})
        out += flatten(n.get("nodes", []), p)
    return out

NODES = []     # one flat list across both books
for bi, f in enumerate(BOOKS, start=1):
    tree = await md_to_tree(f, if_add_node_summary="no", if_add_node_text="yes",
                            if_add_node_id="yes", model=f"gemini/{CHAT_MODEL}")
    flat = flatten(tree["structure"])
    for n in flat:
        n["uid"] = f"B{bi}-{n['node_id']}"
        n["book"] = f"Book{bi}"
        n["path"] = f"Book{bi} > {n['path']}"
        n["chars"] = len(n["text"])
    NODES += flat
    print(f"{f}: {len(flat)} nodes, depth up to {max(n['depth'] for n in flat)}, "
          f"median {sorted(n['chars'] for n in flat)[len(flat)//2]} chars, largest {max(n['chars'] for n in flat):,}")
print("Total nodes:", len(NODES))

In [ ]:
# Step 5: one short summary per node. This is the ONLY indexing cost (the tree itself is free).
t0 = time.time()
before = dict(USAGE)
tok_in = tok_out = tok_th = 0
for i, n in enumerate(NODES):
    body = n["text"].strip()
    if len(body) < 60:                       # heading-only node: nothing to summarize
        n["summary"] = ""
        continue
    p = (f"Summarize this section of a book in at most {SUMMARY_WORDS} words, saying what question it "
         f"would answer. Section title: {n['title']}\n\nTEXT:\n{body[:6000]}")
    s, a, b, c = llm(p)
    n["summary"] = s.strip(); tok_in += a; tok_out += b; tok_th += c
    if i % 50 == 0: print(f"  {i}/{len(NODES)}")
INDEX = {"nodes": len(NODES), "seconds": round(time.time() - t0, 1),
         "prompt_tokens": tok_in, "answer_tokens": tok_out, "thought_tokens": tok_th}
print("Index built:", INDEX)
json.dump({"index": INDEX, "nodes": NODES}, open("pi_tree.json", "w"), indent=1)

In [ ]:
# Step 6: the tree the search model sees (titles, ids and summaries; NO node text).
def tree_view():
    lines = []
    for n in NODES:
        indent = "  " * n["depth"]
        s = f" :: {n['summary']}" if n["summary"] else ""
        lines.append(f"{indent}[{n['uid']}] {n['title']}{s}")
    return "\n".join(lines)
TREE = tree_view()
print(f"Tree view: {len(TREE):,} characters (about {len(TREE)//4:,} tokens). It is sent on every question.")
print(TREE[:1200])

In [ ]:
# Step 7: the PageIndex steps for one question.
BY_UID = {n["uid"]: n for n in NODES}

def search_prompt(q):
    return f"""You are given a question and the tree index of two books (Book1 and Book2). Each line is a
section: [node id] title :: summary. Indentation shows nesting.
Pick the sections whose text most likely contains the answer. Choose at most {MAX_NODES} node ids.
If no section looks like it contains the answer, return an empty list.

QUESTION: {q}

TREE:
{TREE}

Reply as JSON: {{"thinking": "<which sections and why>", "node_list": ["B1-0001", ...]}}"""

def answer_prompt(q, nodes):
    # Same instruction text as the A5 notebook; the labels name the node instead of a chunk number.
    ctx = "\n\n---\n\n".join(f"[Source: {n['uid']} | {n['path']}]\n{n['text'][:MAX_NODE_CHARS]}" for n in nodes)
    return f"""You are an assistant answering from supplied context only.

Using the context below, answer the question at the end. If the context does not
contain the answer, say you do not have enough information. Cite the chunk
numbers you used.

CONTEXT:
{ctx}

QUESTION: {q}
"""

def ask_pageindex(q):
    t0 = time.time()
    raw, s_in, s_out, s_th = llm(search_prompt(q), json_mode=True)
    t1 = time.time()
    try:
        picked = json.loads(raw).get("node_list", [])
        thinking = json.loads(raw).get("thinking", "")
    except Exception:
        picked, thinking = re.findall(r"B\d-\d{4}", raw), raw
    picked = [u for u in dict.fromkeys(picked) if u in BY_UID][:MAX_NODES]
    nodes = [BY_UID[u] for u in picked]
    ans, a_in, a_out, a_th = llm(answer_prompt(q, nodes))
    t2 = time.time()
    return {"question": q, "answer": ans, "thinking": thinking, "picked": picked,
            "hits": [{"rank": i + 1, "node": n["uid"], "location": n["path"], "chars": n["chars"],
                      "text": n["text"][:MAX_NODE_CHARS]} for i, n in enumerate(nodes)],
            "search_s": round(t1 - t0, 2), "generation_s": round(t2 - t1, 2),
            "search_prompt_tokens": s_in, "search_answer_tokens": s_out, "search_thought_tokens": s_th,
            "prompt_tokens": a_in, "answer_tokens": a_out, "thought_tokens": a_th,
            "context_chars": sum(min(n["chars"], MAX_NODE_CHARS) for n in nodes)}

r = ask_pageindex("What are the seven parts of a strong prompt?")
print(r["picked"], r["search_s"], r["generation_s"]); print(r["answer"][:500])

In [ ]:
# Step 8: run the ten A5 questions (identical wording to A5) and save everything.
QUESTIONS = [
    ("Q1", "According to the research cited in AI's Missing Playbook, what share of organizations use AI in at least one function, and what share of enterprise AI projects produced no measurable effect on the bottom line?"),
    ("Q2", "What are the seven parts of a strong prompt?"),
    ("Q3", "In the three-step customer operations example, what is the Rolled Throughput Yield and the cost per verified chain completion?"),
    ("Q4", "How many evaluation trials are needed to confirm a 90% First Pass Yield at 95% confidence with a margin of error of plus or minus 5 points, and how many at plus or minus 2 points?"),
    ("Q5", "In the document review ledger, what did the organization budget per month versus actually spend, and what share of the true per-file cost was human review?"),
    ("Q6", "What does \"the machinist, not the machine\" mean as an AI architecture?"),
    ("Q7", "In AI's Missing Playbook, what is the third option besides build and buy, and what rule governs it?"),
    ("Q8", "What must the ninety-day proof point deliver, and which level of the AI Maturity Self-Assessment does achieving it correspond to?"),
    ("Q9", "What chunk size, in characters or tokens, does AI's Missing Playbook recommend for a RAG pipeline?"),
    ("Q10", "What are the three tiers of the Three-Tier Enterprise Data Taxonomy?"),
]
results = []
for qid, q in QUESTIONS:
    r = ask_pageindex(q); r["id"] = qid; results.append(r)
    print(f"{qid} done | picked {r['picked']} | search {r['search_s']}s, answer {r['generation_s']}s, "
          f"{r['search_prompt_tokens'] + r['prompt_tokens']:,} prompt tokens")

settings = dict(method="PageIndex-style tree search", chat_model=CHAT_MODEL, max_nodes=MAX_NODES,
                summary_words=SUMMARY_WORDS, max_node_chars=MAX_NODE_CHARS, n_nodes=len(NODES),
                files=BOOKS, pageindex_version=md_.version("pageindex"), run_at=time.strftime("%Y-%m-%d %H:%M"),
                tree_chars=len(TREE))
json.dump({"settings": settings, "index": INDEX, "results": results}, open("pi_results.json", "w"), indent=2)

with open("pi_retrieval_dump.md", "w") as f:
    f.write(f"# PageIndex retrieval dump\n\nSettings: {settings}\n\n")
    for r in results:
        f.write(f"\n---\n\n## {r['id']}: {r['question']}\n\n")
        f.write(f"**Search model's reasoning:** {r['thinking']}\n\n")
        for h in r["hits"]:
            f.write(f"### Rank {h['rank']} | node {h['node']} | {h['chars']} chars | {h['location']}\n\n")
            f.write("```text\n" + h["text"] + "\n```\n\n")
        if not r["hits"]: f.write("(no nodes selected)\n\n")
        f.write(f"### Answer\n\n{r['answer']}\n\n")
        f.write(f"*Search {r['search_s']}s, answer {r['generation_s']}s; search {r['search_prompt_tokens']} + answer "
                f"{r['prompt_tokens']} prompt tokens; {r['search_answer_tokens'] + r['answer_tokens']} output tokens*\n")
for name in ("pi_results.json", "pi_retrieval_dump.md", "pi_tree.json"):
    files.download(name)
print("Saved and downloaded pi_results.json, pi_retrieval_dump.md, pi_tree.json")